# MobileNetV2 keypoint regressor — landing base (5 keypoints)

> **Project authored by Claude (Anthropic, Opus 4.8)** for the `a6_gaivotas_ccd2026` project.

Same recognition scheme as the YOLO-pose model: **1 center + 4 corners** of the base.
Direct keypoint regression (no bbox, one base per image) on a MobileNetV2 backbone.

Input: Label Studio export (one native LS `.json` + an `images/` folder), attached as a Kaggle dataset.

Metrics (regression, NOT detection — there is no YOLO-style mAP50 here):
- mean normalized keypoint error
- PCK@0.05
- OKS-AP@0.5 (pose-style proxy, comparable in spirit to mAP50)

Before running: Settings -> Accelerator -> **GPU**, and **Internet ON** (ImageNet weights).

In [ ]:
import os, json, math, random
from pathlib import Path
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
NUM_KPTS = 5            # center + 4 corners
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
# Locate the Label Studio json and the images/ folder inside the attached dataset
input_root = Path('/kaggle/input')
ls_json = next(input_root.rglob('*.json'))
images_root = next(p for p in input_root.rglob('images') if p.is_dir())
print('LS JSON   :', ls_json)
print('images dir:', images_root)
print('n images  :', sum(1 for _ in images_root.glob('*')))

In [ ]:
# Keypoint order: [center, TL, TR, BR, BL]. Corners share the label 'corner' in LS
# and are unordered, so assign each to a fixed slot by its angle around the center.
CANON = [math.atan2(-1, -1), math.atan2(-1, 1), math.atan2(1, 1), math.atan2(1, -1)]  # TL,TR,BR,BL

def ang_dist(a, b):
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

def assign_corners(center, corners):
    cx, cy = center
    pairs = []
    for ci, (x, y) in enumerate(corners):
        a = math.atan2(y - cy, x - cx)
        for si, ref in enumerate(CANON):
            pairs.append((ang_dist(a, ref), ci, si))
    pairs.sort()
    slots = [None] * 4
    uc, us = set(), set()
    for _, ci, si in pairs:
        if ci in uc or si in us:
            continue
        slots[si] = corners[ci]; uc.add(ci); us.add(si)
    out = []
    for s in slots:
        out.append((s[0], s[1], 1.0) if s is not None else (0.0, 0.0, 0.0))
    return out

with open(ls_json) as f:
    tasks = json.load(f)

samples = []   # (image_path, kpts[5,2] in [0,1], vis[5])
skipped = 0
for task in tasks:
    name = Path(task.get('data', {}).get('image', '')).name
    src = next(images_root.rglob(name), None)
    anns = task.get('annotations', [])
    if src is None or not anns:
        skipped += 1; continue
    res = anns[0].get('result', [])
    kc = [r for r in res if r.get('type') == 'keypointlabels'
          and r['value'].get('keypointlabels', [''])[0] == 'center']
    kk = [r for r in res if r.get('type') == 'keypointlabels'
          and r['value'].get('keypointlabels', [''])[0] == 'corner']
    if not kc:
        skipped += 1; continue
    c = kc[0]['value']
    center = (c['x'] / 100.0, c['y'] / 100.0)              # LS stores percent (0-100)
    corners = [(r['value']['x'] / 100.0, r['value']['y'] / 100.0) for r in kk]
    slots = assign_corners(center, corners)
    kpts = np.array([[center[0], center[1]]] + [[x, y] for x, y, v in slots], dtype=np.float32)
    vis = np.array([1.0] + [v for x, y, v in slots], dtype=np.float32)
    samples.append((str(src), kpts, vis))

print('samples:', len(samples), '| skipped:', skipped)

In [ ]:
random.shuffle(samples)
k = int(len(samples) * 0.8)
train_s, val_s = samples[:k], samples[k:]
print('train', len(train_s), '| val', len(val_s))

In [ ]:
HFLIP = [0, 2, 1, 4, 3]   # mirror L-R: center, TL<->TR, BR<->BL
VFLIP = [0, 4, 3, 2, 1]   # mirror U-D: center, TL<->BL, TR<->BR

def augment(img, pts, vis, max_rot=12.0, scale_rng=(0.8, 1.2), trans=0.06):
    H, W = img.shape[:2]
    if random.random() < 0.5:
        img = np.clip(img.astype(np.float32) * random.uniform(0.7, 1.3)
                      + random.uniform(-20, 20), 0, 255).astype(np.uint8)
    ang = random.uniform(-max_rot, max_rot)
    sc = random.uniform(*scale_rng)
    M = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), ang, sc)
    M[0, 2] += random.uniform(-trans, trans) * W
    M[1, 2] += random.uniform(-trans, trans) * H
    img = cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)
    homog = np.hstack([pts * [W, H], np.ones((len(pts), 1), np.float32)])
    pts = (M @ homog.T).T / [W, H]
    if random.random() < 0.5:
        img = img[:, ::-1]; pts[:, 0] = 1.0 - pts[:, 0]; pts = pts[HFLIP]; vis = vis[HFLIP]
    if random.random() < 0.3:
        img = img[::-1]; pts[:, 1] = 1.0 - pts[:, 1]; pts = pts[VFLIP]; vis = vis[VFLIP]
    inb = ((pts[:, 0] >= 0) & (pts[:, 0] <= 1) & (pts[:, 1] >= 0) & (pts[:, 1] <= 1)).astype(np.float32)
    return img, pts, vis * inb

class KptSeq(keras.utils.Sequence):
    def __init__(self, data, batch=16, size=IMG_SIZE, train=True):
        self.data = list(data); self.batch = batch; self.size = size; self.train = train
    def __len__(self):
        return max(1, math.ceil(len(self.data) / self.batch))
    def on_epoch_end(self):
        if self.train:
            random.shuffle(self.data)
    def __getitem__(self, idx):
        bs = self.data[idx * self.batch:(idx + 1) * self.batch]
        X = np.zeros((len(bs), self.size, self.size, 3), np.float32)
        Y = np.zeros((len(bs), NUM_KPTS * 3), np.float32)
        for i, (path, kpts, vis) in enumerate(bs):
            img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            pts, vv = kpts.copy(), vis.copy()
            if self.train:
                img, pts, vv = augment(img, pts, vv)
            img = cv2.resize(img, (self.size, self.size))
            X[i] = keras.applications.mobilenet_v2.preprocess_input(img.astype(np.float32))
            pc = np.clip(pts, 0.0, 1.0)
            Y[i] = np.stack([pc[:, 0], pc[:, 1], vv], axis=1).reshape(-1)
        return X, Y

In [ ]:
def masked_mse(y_true, y_pred):
    yt = tf.reshape(y_true, (-1, NUM_KPTS, 3))
    yp = tf.reshape(y_pred, (-1, NUM_KPTS, 2))
    vis = yt[..., 2:3]
    se = tf.square(yt[..., :2] - yp) * vis
    return tf.reduce_sum(se) / (tf.reduce_sum(vis) * 2.0 + 1e-6)

def kpt_err(y_true, y_pred):
    yt = tf.reshape(y_true, (-1, NUM_KPTS, 3))
    yp = tf.reshape(y_pred, (-1, NUM_KPTS, 2))
    vis = yt[..., 2]
    d = tf.sqrt(tf.reduce_sum(tf.square(yt[..., :2] - yp), -1) + 1e-9)
    return tf.reduce_sum(d * vis) / (tf.reduce_sum(vis) + 1e-6)

def build_model(train_backbone):
    base = keras.applications.MobileNetV2(include_top=False, weights='imagenet',
                                          input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')
    base.trainable = train_backbone
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)          # keep BN in inference mode (recommended for fine-tune)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_KPTS * 2, activation='sigmoid')(x)
    return keras.Model(inp, out), base

model, base = build_model(train_backbone=False)
model.compile(keras.optimizers.Adam(1e-3), loss=masked_mse, metrics=[kpt_err])
model.summary()

In [ ]:
train_gen = KptSeq(train_s, batch=16, train=True)
val_gen = KptSeq(val_s, batch=16, train=False)

def callbacks():
    return [
        keras.callbacks.ReduceLROnPlateau(monitor='val_kpt_err', mode='min', factor=0.5, patience=8, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_kpt_err', mode='min', patience=25, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('/kaggle/working/best.keras', monitor='val_kpt_err', mode='min', save_best_only=True),
    ]

# Phase 1: frozen backbone (train the head)
h1 = model.fit(train_gen, validation_data=val_gen, epochs=40, callbacks=callbacks())

# Phase 2: fine-tune the whole backbone at a lower LR
base.trainable = True
model.compile(keras.optimizers.Adam(1e-4), loss=masked_mse, metrics=[kpt_err])
h2 = model.fit(train_gen, validation_data=val_gen, epochs=120, callbacks=callbacks())

In [ ]:
import matplotlib.pyplot as plt

def cat(a, b, key):
    return a.history[key] + b.history[key]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(cat(h1, h2, 'loss'), label='train'); ax[0].plot(cat(h1, h2, 'val_loss'), label='val')
ax[0].axvline(len(h1.history['loss']) - 0.5, ls='--', c='gray'); ax[0].set_title('masked MSE loss'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(cat(h1, h2, 'kpt_err'), label='train'); ax[1].plot(cat(h1, h2, 'val_kpt_err'), label='val')
ax[1].axvline(len(h1.history['loss']) - 0.5, ls='--', c='gray'); ax[1].set_title('mean normalized kpt error'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()
print('best val kpt error: %.4f' % min(cat(h1, h2, 'val_kpt_err')))

In [ ]:
# Regression metrics on val. NOT YOLO mAP50 (no detection/confidence).
# OKS-AP@0.5 below is a pose-style proxy comparable in spirit.
KPT_K = np.array([0.05, 0.07, 0.07, 0.07, 0.07])   # per-kpt tolerance (center tighter)

def predict(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = keras.applications.mobilenet_v2.preprocess_input(cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32))
    p = model.predict(x[None], verbose=0)[0].reshape(NUM_KPTS, 2)
    return p, img.shape[:2]

errs, pck, oks = [], [], []
for path, kpts, vis in val_s:
    pred, _ = predict(path)
    m = vis > 0
    if m.sum() == 0:
        continue
    d = np.linalg.norm(kpts - pred, axis=1)          # normalized distance
    errs.append(d[m].mean())
    pck.append((d[m] < 0.05).mean())
    vp = kpts[m]
    area = max((vp[:, 0].max() - vp[:, 0].min()) * (vp[:, 1].max() - vp[:, 1].min()), 1e-4)
    e = np.exp(-(d[m] ** 2) / (2 * area * KPT_K[m] ** 2 + 1e-9))
    oks.append(e.mean())

errs, pck, oks = np.array(errs), np.array(pck), np.array(oks)
print('val mean normalized kpt error : %.4f' % errs.mean())
print('val PCK@0.05                  : %.3f' % pck.mean())
print('val mean OKS                  : %.3f' % oks.mean())
print('val OKS-AP@0.5 (proxy mAP50)  : %.3f' % (oks >= 0.5).mean())

In [ ]:
import matplotlib.pyplot as plt
COL = [(0, 255, 0), (255, 0, 0), (255, 128, 0), (0, 0, 255), (255, 0, 255)]  # center, TL,TR,BR,BL

n = min(6, len(val_s))
fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for ax, (path, kpts, vis) in zip(axs.ravel(), val_s[:n]):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    pred, _ = predict(path)
    for j in range(NUM_KPTS):
        if vis[j] > 0:
            cv2.circle(img, (int(kpts[j, 0] * W), int(kpts[j, 1] * H)), 6, COL[j], 2)   # GT: ring
        cv2.circle(img, (int(pred[j, 0] * W), int(pred[j, 1] * H)), 5, COL[j], -1)       # pred: filled
    ax.imshow(img); ax.axis('off')
fig.suptitle('GT (ring) vs Pred (filled) - center=green, corners=TL/TR/BR/BL')
plt.tight_layout(); plt.show()

In [ ]:
best = keras.models.load_model('/kaggle/working/best.keras',
                               custom_objects={'masked_mse': masked_mse, 'kpt_err': kpt_err})
best.save('/kaggle/working/mobilenetv2_pose.keras')

conv = tf.lite.TFLiteConverter.from_keras_model(best)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
open('/kaggle/working/mobilenetv2_pose_fp16.tflite', 'wb').write(conv.convert())

for f in sorted(Path('/kaggle/working').glob('mobilenetv2_pose*')):
    print(f, '%.2f MB' % (f.stat().st_size / 1e6))